# Jupyter/IPython magics

> **Install for this page:**
> `pip install "scope-profiler[notebook,pproc,line-profiler]"` enables every
> example. `notebook` alone covers the basic magics, `pproc` adds plots and
> DataFrames, and `line-profiler` enables `%%scope_line`.

`scope_profiler.ipython_magics` adds eleven magics to a notebook or IPython
session, so a quick "how long is this cell/line taking, and where" doesn't
need the `ProfileManager.session()`/`region()` boilerplate spelled
out by hand each time. It is a thin adapter, not a second implementation:
every magic calls straight into the same API the [HDF5 output & post-processing from Python](/scope-profiler/guide/hdf5-and-python-api/)
page describes - `ProfileManager.session`/`region`,
`ProfilingResults.print_summary`/`to_dataframe`, `read_h5`, the
`diff_rows`/`print_diff_table` functions behind `scope-profiler diff`, and
`prof_export`/`speedscope_export`.

```text
%%scope / %scope_timeit / %%scope_line / %%scope_recursive / %%scope_agg
    |  ProfileManager.session(return_results=True)
    v
ProfilingResults  ---kept in memory, keyed by name---   <-- %scope_load (read_h5)
    |
    +-- print_summary()      (the recording magics, %scope_last)
    +-- to_dataframe() / to_events_dataframe()   (%scope_df)
    +-- plot_durations()     (-p / --plot)
    +-- diff_rows() + print_diff_table()   (%scope_compare)
    +-- export_prof() / export_speedscope()   (%scope_export)
```

A rule of thumb for which recording magic to reach for:

| You want | Magic |
|---|---|
| Time this cell as one block | `%%scope` |
| Time one statement, repeatedly | `%scope_timeit` |
| Find the slow *line* in a function | `%%scope_line` |
| Find the slow *function*, with nothing instrumented | `%%scope_recursive` |
| Time a region entered millions of times | `%%scope_agg` |

Every cell below runs live in this notebook. The examples work on a tiny toy
workload (`time.sleep` stands in for real work, so the timings are stable on
any machine), and nothing is written outside a scratch directory.

## Installing and loading

```bash
pip install "scope-profiler[notebook]"
```

This is a separate extra from the base install (it pulls in `IPython`);
`pip install scope-profiler` alone is unaffected. `-p`/`--plot` additionally
needs the `pproc` extra, exactly as `scope-profiler plot` does.

Load the extension once per kernel:

In [ ]:
%load_ext scope_profiler.ipython_magics

In [ ]:
import os
import tempfile
import time

from scope_profiler import ProfileManager

os.chdir(tempfile.mkdtemp(prefix="scope-profiler-magics-"))
os.makedirs("results")


def solve(problem):
    time.sleep(0.002)  # stands in for real work
    return sum(problem)


def advance(step):
    return step * 2


def solve_naive(problem):
    for _ in problem:
        time.sleep(0.0005)
    return sum(problem)


def solve_optimized(problem):
    time.sleep(0.001)
    return sum(problem)


problem = list(range(10))

## `%%scope` - profile a cell

In [ ]:
%%scope solve
result = solve(problem)

Runs the cell inside a profiling region named `solve` (the name is optional;
it defaults to `"cell"`) and prints the region summary table immediately
after. Nothing is written to disk - the result lives in the kernel's memory
for the rest of the session, stored under that name for `%scope_last` and
`%scope_compare` to refer back to.

Options:

- `-q`/`--quiet` - suppress the printed table (still records the run).
- `-p`/`--plot` - also show a duration bar chart (`plot_durations`).
- `--include PATTERN` - regex restricting which regions the table/plot show,
  same matching `ProfilingResults.get_regions` uses.

To profile sub-parts of the cell separately, nest `ProfileManager.region`
calls inside it as usual - `%%scope` only adds the outer region and the
in-memory session, everything nested inside works exactly as it does in a
script. With `--include`, only the matching regions are shown:

In [ ]:
%%scope --include load|process
with ProfileManager.region("load"):
    time.sleep(0.002)

with ProfileManager.region("process"):
    time.sleep(0.001)

`-p` adds the duration bar chart, and `-q` records the run without printing anything:

In [ ]:
%%scope -p
result = solve(problem)

In [ ]:
%%scope -q quiet_run
result = solve(problem)

## `%scope_timeit` - time a statement

In [ ]:
%scope_timeit -n 20 solve(problem)

Runs the given statement `n` times (default 7) inside a profiling region
named `"timeit"` and prints calls/total/avg/min/max. It answers the same
question as the built-in `%timeit`, but through scope-profiler's own region
timer and table instead of the `timeit` module - useful when you want the
result to be comparable with `%%scope`/`%scope_compare` output. The run is
stored as `timeit`, so `%scope_last timeit -p` shows it as a bar chart.

`-q` suppresses the printed table, same as for `%%scope`.

## `%%scope_line` - line-by-line profiling for a cell

In [ ]:
%%scope_line
from scope_profiler import ProfileManager

@ProfileManager.profile("compute")
def compute(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

compute(100_000)

Runs the cell with `use_line_profiler=True` and prints per-line hit
counts/time for whatever the cell itself hands to line_profiler, the same as
running the cell as a script (see [Line profiler](/scope-profiler/guide/line-profiler/) and
`examples/ex_line_profiling.py`). Needs the `line-profiler` extra:

```bash
pip install "scope-profiler[line-profiler]"
```

Prefer decorating the function you care about with `@ProfileManager.profile`,
as above - it registers cleanly regardless of where it's called from. A bare
`with ProfileManager.region(...):` at the cell's top level also
works, but line_profiler's own handling of a module-level code object
containing a nested `def` can cut the printed table off at that `def` --
the same limitation a plain script hits, not something specific to the
magic.

`-q` suppresses the region summary table, `-Q` suppresses the line-by-line
tables; both can be combined.

## `%%scope_recursive` - profile a cell with nothing instrumented

In [ ]:
%%scope_recursive
def inner(values):
    return sum(value * value for value in values)


def outer(rounds):
    return [inner(range(2_000)) for _ in range(rounds)]


result = outer(2)

Records **every Python call the cell makes** as its own region, named
`<module>.<qualname>`, with no decorators and no `region` blocks --
`recursive_profile=True`, the same thing `scope-profiler run` does to a
script. The summary comes back sorted by total time, so the answer to "where
did this cell's time actually go?" is the top of the table, as above.

This is the magic to reach for *first*, before you know what to instrument:
use it to find the hot function, then wrap that one in `%%scope`/`%%scope_line`
for a cheap, focused measurement.

Two things to know, both inherited from how recursive profiling works in a
script rather than from the magic:

- Tracing is not limited to your own functions - library internals called
  from the cell are recorded too, and each traced call carries per-call
  overhead. Keep the cell small and use `--include` to filter the table.
- The cell is run with `exec` rather than IPython's own execution, so a
  trailing expression is not echoed as `Out[n]`. Assignments land in the
  notebook namespace exactly as usual.

A cell that raises is reported the way any other failing cell is, and the
profile of everything up to the failure is still recorded and printed --
which is often exactly what you want to see when a run dies halfway:

In [ ]:
%%scope halfway
with ProfileManager.region("step 1"):
    time.sleep(0.001)
raise RuntimeError("the run died after step 1")

That holds for `%%scope` (as above) and `%%scope_agg` too.

## `%%scope_agg` - aggregation mode for very hot regions

In [ ]:
%%scope_agg
for step in range(20_000):
    with ProfileManager.region("step"):
        advance(step)

Runs the cell with `aggregation_mode=True`: each region keeps its count,
inclusive total, min/max and exclusive total, and no per-call timeline at
all. A region entered a million times therefore costs a few numbers instead
of a million timestamps, which is what makes such a cell measurable at all
(the cell above uses 20 000 steps only to stay quick).

The trade-off is that the per-call event data is gone: the Gantt chart,
`%scope_df --events` and the timeline exports have nothing to show. The
summary table, `%scope_compare` and `%scope_df` (region rows) work as usual.
Aggregation mode cannot be combined with line, GPU, NVTX or LIKWID timing.

## `%scope_load` - bring in a run from outside the notebook

Profiling data from an MPI job or a `scope-profiler run` is just an HDF5
file. The two files below stand in for such runs, written by plain sessions
into `results/`:

In [ ]:
for file_name, work in [("run_128ranks.h5", solve), ("before.h5", solve_naive)]:
    with ProfileManager.session(file_path=f"results/{file_name}", verbose=False):
        with ProfileManager.region("solve"):
            work(problem)

In [ ]:
%scope_load results/run_128ranks.h5
%scope_load -n baseline results/before.h5

Reads an HDF5 file with `read_h5` and stores it under the same names the
other magics use - so a run from an MPI job, a `scope-profiler run`, or a
colleague becomes directly comparable with what you just measured in a cell:

In [ ]:
%%scope candidate
result = solve(problem)

In [ ]:
%scope_compare run_128ranks candidate

The name defaults to the file's stem; `-n`/`--name` overrides it and `-q`
skips the summary table.

## `%scope_df` - a run as a pandas DataFrame

In [ ]:
df = %scope_df
df.nlargest(5, "total_duration")

Returns the recorded run as a DataFrame (`to_dataframe`), so it both renders
as a table on its own and can be captured for analysis, grouping or plotting
in the notebook - everything [2. Post-processing profiling data](/scope-profiler/guide/02-postprocessing/) does with a
file, on the run you just measured. Needs pandas (the `pproc` extra).

- `-n`/`--name` - which recorded run (default: the most recent).
- `--per-rank` - one row per (region, rank) instead of one aggregated row.
- `--events` - one row per recorded call (`to_events_dataframe`) instead of
  one per region. An aggregation-mode run records no per-call events, so this
  comes back empty with a note saying why.
- `--include`/`--exclude` - regex region filters.

In [ ]:
%scope_df --events -n timeit

## `%scope_last` - reprint a previous run

With no name, it reprints the most recently recorded run:

In [ ]:
%scope_last

or a specific one, by name:

In [ ]:
%scope_last solve

Reprints the summary table for a run recorded earlier by `%%scope` or
`%scope_timeit`, without re-running anything. Takes the same `-p`/`--plot`
and `--include` options as `%%scope`. Useful after scrolling past a result,
or to re-render it with a different `--include` filter.

## `%scope_compare` - compare two runs

In [ ]:
%%scope baseline
result = solve_naive(problem)

In [ ]:
%%scope candidate
result = solve_optimized(problem)

In [ ]:
%scope_compare baseline candidate

Aligns the two runs' regions by name and prints one row per region with the
change in a chosen metric - the same table `scope-profiler diff` prints for
two HDF5 files, here for two in-notebook runs. With no arguments, it compares
the two most recently recorded runs, so `%scope_compare` right after the two
cells above works without naming them:

In [ ]:
%scope_compare

- `--metric` - one of `total`, `avg`, `min`, `max`, `p50`, `p95`, `p99`,
  `imbalance`, `calls` (default `total`).
- `--sort` - `delta`, `pct`, or `name` (default `delta`).

In [ ]:
%scope_compare baseline candidate --metric avg --sort name

## `%scope_export` - export a run to a file

In [ ]:
%scope_export candidate.prof
%scope_export -n baseline baseline.speedscope.json

Writes a previously recorded run to a `.prof` file (openable in `snakeviz` or
any `pstats` viewer) or a speedscope JSON file, via
`scope_profiler.prof_export.export_prof`/
`scope_profiler.speedscope_export.export_speedscope` - the same functions
`scope-profiler export` uses on an HDF5 file, here applied directly to an
in-memory `ProfilingResults`. Format is inferred from `filepath`'s extension
(`--format prof`/`--format speedscope` to force one); `-n`/`--name` selects
which recorded run to export (default: the most recent one); `--include`/
`--exclude` restrict which regions are written. A `.prof` file is written per
rank, so `candidate.prof` ends up as `candidate_rank0.prof`. For `.prof` output,
`--no-call-paths` writes one entry per region instead of one per
`parent > child` path.

In [ ]:
sorted(name for name in os.listdir() if name.endswith((".prof", ".json")))

## `%scope_reset` - drop recorded runs

In [ ]:
%scope_reset baseline
%scope_reset

`%scope_reset baseline` drops just one run, by name; plain `%scope_reset`
clears every recorded run. Recorded runs otherwise accumulate in memory for the life of the kernel.
Useful once `%scope_last`/`%scope_compare` have several names to choose from
and you want a clean slate, or to free a large run's timeline once you are
done with it.

## Tutorial

[7. Notebook magics](/scope-profiler/guide/07-notebook-magics/) walks through all eleven magics against a
small toy workload, including comparing a "before" and "after" version of a
function.